# Model recovery and parameter recovery — RL vs RLWM (PyBADS)

This notebook fits **both agents to all datasets generated by both agents**, compares the models with **BIC**, plots a **model-recovery confusion matrix**, and checks **parameter recovery**. Finally, it fits the original `df_rlwm_clean.csv` participants.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, and `pybads` (**no `numba` or `tqdm`**). If needed, run `pip install pybads` in your environment. [PyBADS documentation](https://acerbilab.github.io/pybads/).

**Important: exact models used to generate these CSVs**
- **RL** has `alphaRL` and `betaRL`, with no forgetting.
- **RLWM** adds `decayRL`, `decayWM`, and `WMweight`. WM learns with rate **1**, and both value tables decay **before** the chosen action is updated.
- **WM capacity is fixed: `K = 3`**, *not* fitted. The choice weight in a set of size `ns` is `WMweight * min(1, K/ns)`.
- Every set starts with zero action values. The observed feedback uses **0 for an incorrect action** and **1 or 2 for a correct action**. Actions in the CSV are 1–3 and are converted to 0–2 for fitting.

**Workflow:** first run with `QUICK_TEST = True`, then set it to `False` for all 100 simulations and all real participants. Saved fits resume automatically after interruption. Results are stored separately for quick vs full analyses.

In [ ]:
# Settings and imports
from pathlib import Path
import math
import zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK_TEST = False        # Set to False for the complete model/parameter recovery
K = 3                    # Fixed WM capacity used by the uploaded RLWM simulations
BASE_SEED = 12345        # Use to reproduce the same fitting results each time you start this notebook

OUTPUT = Path('recovery_results') / ('quick' if QUICK_TEST else 'full') / f'K{K}'

OUTPUT_REALDATA = Path('real_results') / f'K{K}'

PARAM_NAMES = {
    'RL': ['alphaRL', 'betaRL'],
    'RLWM': ['alphaRL', 'betaRL', 'decayRL', 'decayWM', 'WMweight'],
}

BOUNDS = {
    'alphaRL': (0.0001, 1.0),
    'betaRL': (-10.0, 100.0),
    'decayRL': (0.0, 1.0),
    'decayWM': (0.0, 1.0),
    'WMweight': (0.0, 1.0),
}


print('Saving results in:', OUTPUT.resolve())

## 1. Load the datasets

In [ ]:
# Paths of the relevant files
sim_path = Path.cwd() / 'simulated_RL_and_RLWM.csv'
rl_path = Path.cwd() / 'parameters_RL.csv'
rlwm_path = Path.cwd() / 'parameters_RLWM.csv'
realdata_path = Path.cwd() / 'preprocessed_data' / 'df_rlwm_clean.csv'
#
df_sim = pd.read_csv(sim_path)
df_rl_params = pd.read_csv(rl_path)
df_rlwm_params = pd.read_csv(rlwm_path)
df_real = pd.read_csv(realdata_path)

# Only learning trials contribute to likelihood fits here.
df_sim = df_sim[df_sim['trial_mode'] == 'learning'].copy()
df_real = df_real[df_real['trial_mode'] == 'learning'].copy()
true_params = pd.concat([df_rl_params, df_rlwm_params], ignore_index=True)

# Check that the simulation parameters & surrogate data files correspond
assert set(df_sim['simId'].unique()) == set(true_params['simId'].unique())

print('Simulations:', df_sim.groupby('model')['simId'].nunique().to_dict())
print('Trials per simulation:', df_sim.groupby('simId').size().unique().tolist())
print('Real participants:', df_real['subjid'].nunique())
print('Real learning trials:', len(df_real))

## 2. Confusion matrix (BIC model selection)

Rows are **generating agents**, columns are **recovered models**. Each entry is the number of datasets and the percentage within that true-model row. The diagonal indicates successful model recovery.

In [ ]:
# Load recovery results
sim_fits_path = OUTPUT / 'fits_simulations.csv'

sim_fits=pd.read_csv(sim_fits_path)

# Only datasets for which BOTH candidate fits are available can enter the confusion matrix.
complete = sim_fits.groupby('id')['fit_model'].nunique()
complete_ids = complete[complete == 2].index
comp = sim_fits[sim_fits['id'].isin(complete_ids)].copy()
winners = comp.loc[comp.groupby('id')['BIC'].idxmin()].copy()
winners = winners.rename(columns={'fit_model': 'selected_model'})
winners.to_csv(OUTPUT / 'model_recovery_winners.csv', index=False)

labels = ['RL', 'RLWM']
confusion = pd.crosstab(winners['source_model'], winners['selected_model']).reindex(
    index=labels, columns=labels, fill_value=0)
print('Counts:\n', confusion)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(confusion.values, cmap='Blues', vmin=0)
ax.set_xticks(range(2), labels)
ax.set_yticks(range(2), labels)
ax.set_xlabel('Recovered model (minimum BIC)')
ax.set_ylabel('Generating model')
ax.set_title('Model recovery — BIC')
for i in range(2):
    for j in range(2):
        count = confusion.iloc[i, j]
        total = confusion.iloc[i].sum()
        percent = 100 * count / total if total else 0
        ax.text(j, i, f'{count}\n({percent:.0f}%)', ha='center', va='center',
                color='white' if count > confusion.values.max() / 2 else 'black')
fig.tight_layout()
fig.savefig(OUTPUT / 'model_recovery_confusion_matrix.png', dpi=160)
plt.show()

## 3. Parameter recovery: true vs fitted parameters

Use only the **fit of the model that actually generated each dataset**. Compare its fitted parameters with the known generating parameters using a scatterplot and a Pearson correlation. Boundaries, trade-offs between parameters, and limited trial numbers can make some parameters harder to recover than others.

In [ ]:
true_fits = sim_fits[sim_fits['source_model'] == sim_fits['fit_model']].copy()
merged = true_fits.merge(true_params, left_on='id', right_on='simId',
                         how='inner', validate='one_to_one', suffixes=('', '_true'))
summary = []

for model in ('RL', 'RLWM'):
    d = merged[merged['source_model'] == model]
    names = PARAM_NAMES[model]
    ncols = min(3, len(names))
    nrows = math.ceil(len(names) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.2 * nrows),
                            squeeze=False)
    for ax, name in zip(axs.flat, names):
        x = d[name].to_numpy(dtype=float)
        y = d['fit_' + name].to_numpy(dtype=float)
        correlation = np.corrcoef(x, y)[0, 1] if len(x) >= 3 and np.std(x) > 0 and np.std(y) > 0 else np.nan
        rmse = np.sqrt(np.mean((x - y) ** 2)) if len(x) else np.nan
        bias = np.mean(y - x) if len(x) else np.nan
        summary.append({'model': model, 'parameter': name, 'n': len(x),
                        'Pearson_r': correlation, 'RMSE': rmse, 'bias_fit_minus_true': bias})
        ax.scatter(x, y, alpha=0.7, s=35)
        lo, hi = BOUNDS[name]
        ax.plot([lo, hi], [lo, hi], '--', color='gray', linewidth=1)
        ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel='Generating parameter',
               ylabel='Fitted parameter', title=f'{name}\nr={correlation:.2f}; RMSE={rmse:.3f}')
    for ax in list(axs.flat)[len(names):]:
        ax.axis('off')
    fig.suptitle(f'Parameter recovery — {model} (n={len(d)})', fontsize=14)
    fig.tight_layout()
    fig.savefig(OUTPUT / f'parameter_recovery_{model}.png', dpi=160)
    plt.show()

parameter_summary = pd.DataFrame(summary)
parameter_summary.to_csv(OUTPUT / 'parameter_recovery_summary.csv', index=False)
print(parameter_summary.to_string(index=False))

## 4. Real data model comparison

Compare RL and RLWM models fitted on the real data.
Which one works best?



In [ ]:
# Load the real data

real_fits_path =  OUTPUT_REALDATA / 'fits_original.csv'
real_fits =  pd.read_csv(real_fits_path)

real_winners = real_fits.loc[real_fits.groupby('id')['BIC'].idxmin()].copy()
real_winners = real_winners[['id', 'fit_model', 'BIC', 'NLL']].rename(
    columns={'fit_model': 'selected_model'})
real_winners.to_csv(OUTPUT_REALDATA / 'original_model_comparison.csv', index=False)
print('Best model per participant (BIC):')
print(real_winners['selected_model'].value_counts())

In [ ]:
# Compare the model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# Compare the two models for each participant
fits = real_fits.pivot_table(index='id', columns='fit_model',
                             values=['AIC', 'BIC'])

comparison = pd.DataFrame({
    'id': fits.index.astype(str),
    'delta_AIC': (fits['AIC']['RL'] - fits['AIC']['RLWM']).values,
    'delta_BIC': (fits['BIC']['RL'] - fits['BIC']['RLWM']).values
}).dropna()

comparison['best_AIC'] = np.where(comparison.delta_AIC > 0, 'RLWM', 'RL')
comparison['best_BIC'] = np.where(comparison.delta_BIC > 0, 'RLWM', 'RL')

# Long format for plotting
plot_df = comparison.melt(id_vars='id',
                          value_vars=['delta_AIC', 'delta_BIC'],
                          var_name='criterion', value_name='difference')
plot_df['criterion'] = plot_df['criterion'].str.replace('delta_', '')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Distribution of AIC and BIC differences
sns.violinplot(data=plot_df, x='criterion', y='difference',
               hue='criterion', palette='Set2', inner=None,
               legend=False, ax=axes[0])
sns.stripplot(data=plot_df, x='criterion', y='difference',
              color='black', alpha=0.5, jitter=0.18, ax=axes[0])
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set(title='Model comparison',
            xlabel='', ylabel='RL − RLWM (positive favors RLWM)')

# Number of participants best explained by each model
sns.countplot(data=comparison, x='best_BIC', hue='best_BIC',
              order=['RL', 'RLWM'], palette='Set2',
              legend=False, ax=axes[1])
axes[1].set(title='Best-fitting model (BIC)',
            xlabel='Model', ylabel='Number of participants')

for bars in axes[1].containers:
    axes[1].bar_label(bars, padding=3)

plt.tight_layout()
plt.show()

print(f"Participants compared: {len(comparison)}")
print(comparison[['best_AIC', 'best_BIC']]
      .apply(pd.Series.value_counts).fillna(0).astype(int))

comparison.to_csv(OUTPUT / 'real_model_comparison.csv', index=False)


## Interindividual differences: relationship between model parameters and personality traits

Analyse the relation between model parameters and personality traits using real data.

PSS column: perceived stress scale
STAI column: anxiety state scale
BDI column: depression scale

In [ ]:

from scipy.stats import spearmanr

traits = ['STAI', 'BDI', 'PSS']
params = ['fit_alphaRL', 'fit_betaRL', 'fit_decayRL',
          'fit_decayWM', 'fit_WMweight']
outcomes = ['delta_AIC', 'delta_BIC'] + params

# One personality score per participant
scores = (df_real.groupby('subjid', as_index=False)[traits]
          .first().rename(columns={'subjid': 'id'}))
scores['id'] = scores['id'].astype(str)

# Retrieve fitted RLWM parameters
rlwm = real_fits[real_fits.fit_model == 'RLWM'][['id'] + params].copy()
rlwm['id'] = rlwm['id'].astype(str)

# Merge by participant ID
data = comparison.merge(scores, on='id').merge(rlwm, on='id')

# Correlations
results = []
for trait in traits:
    for outcome in outcomes:
        d = data[[trait, outcome]].dropna()
        if len(d) >= 3 and d.nunique().min() > 1:
            r, p = spearmanr(d[trait], d[outcome])
        else:
            r, p = np.nan, np.nan
        results.append([trait, outcome, len(d), r, p])

stats = pd.DataFrame(results, columns=['Trait', 'Variable', 'N', 'rho', 'p'])

# Benjamini-Hochberg correction for multiple comparisons
order = stats['p'].dropna().sort_values().index
if len(order):
    q = stats.loc[order, 'p'].values * len(order) / np.arange(1, len(order) + 1)
    stats.loc[order, 'q_FDR'] = np.minimum(
        1, np.minimum.accumulate(q[::-1])[::-1])

print(stats.sort_values('p').round(4).to_string(index=False))
stats.to_csv(OUTPUT / 'personality_correlations.csv', index=False)

# 1. Correlation heatmap
corr = stats.pivot(index='Trait', columns='Variable', values='rho')
corr = corr.reindex(index=traits, columns=outcomes)

plt.figure(figsize=(12, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag',
            center=0, vmin=-1, vmax=1, linewidths=0.7)
plt.title('Personality traits and model parameters — Spearman correlations')
plt.xlabel('')
plt.ylabel('')
plt.tight_layout()
plt.show()

# 2. Scatterplots for model preference and RLWM parameters
for variables, title in [
    (['delta_AIC', 'delta_BIC'], 'Model preference vs. personality'),
    (params, 'RLWM parameters vs. personality')
]:
    fig, axes = plt.subplots(len(variables), 3,
                             figsize=(13, 3 * len(variables)),
                             squeeze=False)
    colors = sns.color_palette('husl', len(variables))

    for i, variable in enumerate(variables):
        for j, trait in enumerate(traits):
            ax = axes[i, j]

            sns.regplot(data=data, x=trait, y=variable, ax=ax,
                        color=colors[i], ci=None,
                        scatter_kws={'alpha': 0.7, 's': 35},
                        line_kws={'color': 'black', 'linewidth': 1.5})

            if variable.startswith('delta_'):
                ax.axhline(0, color='gray', linestyle='--')

            ax.set(xlabel=trait if i == len(variables)-1 else '',
                   ylabel=variable if j == 0 else '')

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()
